In [1]:
from pathlib import Path
import pickle
import torch
from chggen.pl_data.dataset import CHGNetDataset
from chggen.pl_data.datamodule import CrystDataModule
from chggen.pl_modules.model import CHGGen
from chggen.common.data_utils import get_scaler_from_data_list
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.sampler import SubsetRandomSampler
# from chggen.pl_modules.trainer import Trainer
from torch_geometric.data import Batch

import pytorch_lightning as pl


/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_scaler(dataset, use_prop_scaler = False, 
               scaler_path = None):
    # Load once to compute property scaler
    if scaler_path is None:
        lattice_scaler = get_scaler_from_data_list(
            dataset.cached_data,
            key='scaled_lattice')
        if use_prop_scaler:
            NotImplementedError("Not implemented the multi prop scaler yet.")
    else:
        lattice_scaler = torch.load(
            Path(scaler_path) / 'lattice_scaler.pt')
    return lattice_scaler

In [3]:
dataset = CHGNetDataset(path= '/home/zhongpc/chggen/data/perov_5/test_zpc.csv',
                        name = 'zpc_test',
                        prop_list = ['heat_ref', 'heat_all'],
                        )

lattice_scaler = get_scaler(dataset= dataset)
data_list = [dataset[i] for i in range(len(dataset))]

datamodule = CrystDataModule(train_dataset= dataset,
                             val_dataset= dataset,
                             num_workers=8,
                             batch_size= 7,
                                )

chggen = CHGGen(lattice_scaler= lattice_scaler)


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 248.26it/s]
/home/zhongpc/chggen/chggen/common/data_utils.py:645: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:245.)
  targets = torch.tensor([d[key] for d in data_list])
/home/zhongpc/chggen/chggen/common/data_utils.py:613: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)


CHGNet initialized with 400,438 parameters
CHGNet initialized with 400,438 parameters


In [5]:

for name, param in chggen.named_parameters():
    print(name)
    if "encoder" in name:
        param.requires_grad = False

trainer = pl.Trainer(accelerator = "gpu", 
                     devices = [7],
                     max_epochs= 5,
#                      strategy = 'ddp_find_unused_parameters_true'
                    )

trainer.fit(model= chggen, datamodule= datamodule)


print("Done")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:67: UserWarning: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
  warning_cache.warn(
You are using a CUDA device ('NVIDIA A100 80GB PCIe') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/d

sigmas
type_sigmas
encoder.composition_model.fc.weight
encoder.atom_embedding.embedding.weight
encoder.bond_basis_expansion.rbf_expansion_ag.frequencies
encoder.bond_basis_expansion.rbf_expansion_bg.frequencies
encoder.bond_embedding.weight
encoder.bond_weights_ag.weight
encoder.bond_weights_bg.weight
encoder.angle_basis_expansion.fourier_expansion.frequencies
encoder.angle_embedding.weight
encoder.atom_conv_layers.0.twoBody_atom.mlp_core.layers.0.weight
encoder.atom_conv_layers.0.twoBody_atom.mlp_core.layers.0.bias
encoder.atom_conv_layers.0.twoBody_atom.mlp_core.layers.3.weight
encoder.atom_conv_layers.0.twoBody_atom.mlp_core.layers.3.bias
encoder.atom_conv_layers.0.twoBody_atom.mlp_gate.layers.0.weight
encoder.atom_conv_layers.0.twoBody_atom.mlp_gate.layers.0.bias
encoder.atom_conv_layers.0.twoBody_atom.mlp_gate.layers.3.weight
encoder.atom_conv_layers.0.twoBody_atom.mlp_gate.layers.3.bias
encoder.atom_conv_layers.0.mlp_out.layers.1.weight
encoder.atom_conv_layers.0.mlp_out.layers.1

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]

  | Name           | Type           | Params
--------------------------------------------------
0 | encoder        | CHGNet_encoder | 400 K 
1 | decoder        | GemNetTDecoder | 2.3 M 
2 | fc_mu          | Linear         | 4.2 K 
3 | fc_var         | Linear         | 4.2 K 
4 | fc_num_atoms   | Sequential     | 11.0 K
5 | fc_lattice     | Sequential     | 9.1 K 
6 | fc_composition | Sequential     | 21.2 K
--------------------------------------------------
2.4 M     Trainable params
400 K     Non-trainable params
2.8 M     Total params
11.083    Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/torch_geometric/deprecation.py:22: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


...

/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pytorch_lightning/utilities/data.py:76: UserWarning: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 5. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
  warning_cache.warn(



Validation: 0it [00:00, ?it/s]
Validation DataLoader 0:  88%|███████████████████████████████████████████████████████████████████████████████▋           | 7/8 [00:00<00:00, 12.36it/s]
Epoch 0: 100%|█| 8/8 [00:02<00:00,  2.90it/s, v_num=0, train_loss_step=27.20, train_natom_loss_step=2.480, train_lattice_loss_step=0.651, train_coord_l
Epoch 1: 100%|█| 8/8 [00:01<00:00,  4.74it/s, v_num=0, train_loss_step=269.0, train_natom_loss_step=1.230, train_lattice_loss_step=4.730, train_coord_l
Validation: 0it [00:00, ?it/s]
Validation DataLoader 0:  88%|███████████████████████████████████████████████████████████████████████████████▋           | 7/8 [00:00<00:00, 15.01it/s]
Epoch 1: 100%|█| 8/8 [00:02<00:00,  2.79it/s, v_num=0, train_loss_step=269.0, train_natom_loss_step=1.230, train_lattice_loss_step=4.730, train_coord_l
Epoch 2: 100%|█| 8/8 [00:01<00:00,  5.26it/s, v_num=0, train_loss_step=23.10, train_natom_loss_step=0.449, train_lattice_loss_step=0.133, train_coord_l
Validation: 0it [00:00, ?

Validation DataLoader 0:  88%|███████████████████████████████████████████████████████████████████████████████▋           | 7/8 [00:00<00:00, 13.49it/s]
Epoch 4: 100%|█| 8/8 [00:03<00:00,  2.62it/s, v_num=0, train_loss_step=46.20, train_natom_loss_step=0.00449, train_lattice_loss_step=0.286, train_coord
Epoch 4: 100%|█| 8/8 [00:03<00:00,  2.61it/s, v_num=0, train_loss_step=46.20, train_natom_loss_step=0.00449, train_lattice_loss_step=0.286, train_coord

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|█| 8/8 [00:03<00:00,  2.51it/s, v_num=0, train_loss_step=46.20, train_natom_loss_step=0.00449, train_lattice_loss_step=0.286, train_coord
Done


# save the model

In [21]:
torch.save(chggen.state_dict(), './saved_model.pth')

# save the lattice_scaler

In [ ]:
torch.save(lattice_scaler, './lattice_scaler.pt')

In [22]:
model = torch.load('./saved_model.pth')

In [27]:
model = CHGGen()
model.load_state_dict(torch.load('./saved_model.pth'))

CHGNet initialized with 400,438 parameters
CHGNet initialized with 400,438 parameters


<All keys matched successfully>

In [32]:
model.lattice_scaler = lattice_scaler
model.lattice_scaler

StandardScalerTorch(means: [2.4322593212127686, 2.4322593212127686, 2.4322593212127686, 90.0, 90.0, 90.0], stds: [0.19765666127204895, 0.19765666127204895, 0.19765666127204895, 9.999999747378752e-06, 9.999999747378752e-06, 9.999999747378752e-06])